In [19]:
import pandas as pd
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

In [20]:
def load_and_explore_dataset(file_path):
        print("-" * 60)
        print("LOAD AND EXPLORE DATE")
        print("-" * 60)

        df = pd.read_csv(file_path)
        print('shape of the dataset')
        print(df.shape)
        print('\ncheck missing')
        print(df.isnull().sum())
        print('\nFirst five row')
        print(df.head())
        print('\nDescriptive statistics')
        print(df.describe())
        print('\ndataset info')
        print(df.info())
        print('\nExited Distribution')
        print(df['Exited'].value_counts())
        print('\nExited Percentage Distribution')
        print(df['Exited'].value_counts(normalize=True))

        return df

In [21]:
#identifying features to use 
def identify_features(df):
    print("-" * 60)
    print("IDENTIFY FEATURES")
    print("-" * 60)

    #feature to drop
    feature_to_drop = [
        "CustomerId",
        "Surname"
    ]

    #numerical columns
    num_features = [
        "CreditScore",
        "Age",
        "Tenure",
        "EstimatedSalary",
        "Balance",
        "NumOfProducts",
        "ServiceRating"
    ]

    #category features
    cat_features = [
        "Geography",
        "Gender",
        "HasCrCard",
        "IsActiveMember"
    ]

    print("\nFeatures to drop:", feature_to_drop)
    print("\nNumerical features:", num_features)
    print("\nCategorical features:", cat_features)

    return feature_to_drop, num_features, cat_features

In [22]:
#preparing the data
def prepare_data(df, num_features, cat_features, feature_to_drop):
    print("-" * 60)
    print("PREPARING DATA")
    print("-" * 60)

    #drop unrelevant data
    df_model = df.drop(columns=feature_to_drop)

    #specify label
    #feature to use 
    x = df_model.drop("Exited", axis=1)

    #target to use
    y = df_model["Exited"]

    print("\nFeature Shape:", x.shape)
    print("\nTarget Shape:", y.shape)
    print("\nFeature columns list:", list(x.columns))

    return x, y, list(x.columns)

In [23]:
#creating pipeline
def create_preprocessing_pipeline(num_features, cat_features):
    print("-" * 60)
    print("CREATE PREPROCESSING PIPELINE")
    print("-" * 60)

    #num columns
    numerical_pipeline = Pipeline([
        ("scaler", StandardScaler())
    ])

    #category features
    category_pipeline = Pipeline([
        ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
    ])

    #column preprocessing
    preprocessing = ColumnTransformer([
        ("num", numerical_pipeline, num_features),
        ("cat", category_pipeline, cat_features)
    ])

    return preprocessing

In [24]:
def split_data(x, y, test_size=0.2, random_state=42):
    print("-" * 60)
    print("SPLITTING THE DATA")
    print("-" * 60)

    # x and y train and test
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=test_size, random_state=random_state, stratify=y
    )

    print("\nTraining set size:", x_train.shape[0])
    print("\nTest set size:", x_test.shape[0])

    print("\nTraining churn distribution:", y_train.value_counts())
    print("\nTest churn distribution:", y_test.value_counts())

    return x_train, x_test, y_train, y_test

In [25]:
def create_pipeline(preprocessor):
    print("\n" + "-" * 60)
    print("CREATING PIPELINE")
    print("-" * 60)

    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42,
            class_weight="balanced",
            solver="lbfgs"
        ))
    ])

    print("\nModel Pipeline created successfully")

    return model_pipeline

In [26]:
def train(model_pipeline, x_train, y_train):
    print("\n" + "-" * 60)
    print("TRAINING MODEL")
    print("-" * 60)

    model_pipeline.fit(x_train, y_train)

    print("\nModel train successfully")

    #get name steps from preprocessing
    
    # num_features = model_pipeline.named_steps["preprocessor"].transformers_[0][2]
    # cat_features = model_pipeline.named_steps["preprocessor"].transformers_[1][2]

    # #get name steps from onehot encoder
    # onehot_encoder = model_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
    # cat_feature_name = onehot_encoder.get_feature_names_out(cat_features)

    # all_feature_names = list(num_features) + list(cat_features)

    #get name steps from preprocessing
    all_feature_names = model_pipeline.named_steps[
            "preprocessor"
    ].get_feature_names_out()

    #coefficient
    coefficient = model_pipeline.named_steps["classifier"].coef_[0]
    print("\nTop 10 most importance features (by coefficient magnitude)")
    feature_importance = pd.DataFrame({
        "feature": all_feature_names,
        "coefficient": coefficient
    }).sort_values("coefficient", key=abs, ascending=False)
    print(feature_importance.head(10).to_string(index=False))
    # except Exception as e:
    #     print(f"Cannot extract feature name {e}")

    return model_pipeline 

In [27]:
def evaluate_model(model_pipeline, x_train, x_test, y_train, y_test):
    print("\n" + "-" * 60)
    print("EVALUATE MODEL")
    print("-" * 60)

    #predicted model
    y_train_pred = model_pipeline.predict(x_train)
    y_test_pred = model_pipeline.predict(x_test)

    #predicted model proba
    y_train_prob = model_pipeline.predict_proba(x_train)[:,1]
    y_test_prob = model_pipeline.predict_proba(x_test)[:,1]

    #metrics 
    y_train_acc_score = accuracy_score(y_train, y_train_pred)
    y_test_acc_score = accuracy_score(y_test, y_test_pred)

    y_train_roc_auc_score = roc_auc_score(y_train, y_train_prob)
    y_test_roc_auc_score = roc_auc_score(y_test, y_test_prob)

    print("\nMETRICS EVALUATION")
    print(f"\nTrain accuracy score: {float(y_train_acc_score):.4f}")
    print(f"\nTest accuracy score: {float(y_test_acc_score):.4f}")
    print(f"\nTrain ROC score: {float(y_train_roc_auc_score):.4f}")
    print(f"\nTest ROC score: {float(y_test_roc_auc_score):.4f}")

    print("\nCLASSIFICATION REPORT ON TRAIN SIZE")
    print("\nClassififcation report\n", classification_report(y_train, y_train_pred, target_names=["Retained", "Churned"]))

    print("\nCLASSIFICATION REPORT ON TEST SIZE")
    print("\nClassififcation report\n", classification_report(y_test, y_test_pred, target_names=["Retained", "Churned"]))

    #cross validation
    cv_score = cross_val_score(model_pipeline, x_train, y_train, cv=5, scoring="roc_auc")
    print("\nCross Validation(5-fold)")
    print(f" ROC AUC: {cv_score}")
    print(f" Cross validation mean: {cv_score.mean():.4f}")
    print(f" Cross validation std: {cv_score.std():.4f}")

    metrics = {
        "y_train_acc_score": y_train_acc_score,
        "y_test_acc_score": y_test_acc_score,
        "y_train_roc_auc_score": y_train_roc_auc_score,
        "y_test_roc_auc_score": y_test_roc_auc_score,
        "y_train_pred": y_train_pred,
        "y_test_pred": y_test_pred,
        "y_train_prob": y_train_prob,
        "y_test_prob": y_test_prob,
        "cv_score": cv_score
    }

    return metrics

In [28]:
def save_model_artifact(model_pipeline, featured_columns):
    print("\n" + "-" * 60)
    print("SAVING MODEL ARTIFACT")
    print("-" * 60)

    joblib.dump(model_pipeline, "bank_customer_churned.pkl")
    print("Bank customer churn model saved successfuly")

    joblib.dump(featured_columns, "bank_customer_churned_featured_columns.pkl")
    print("Feature columns saved successfuly")

    print("\n" + "-" * 60)
    print("ALL MODEL ARTIFACT SAVED")
    print("-" * 60)

In [44]:
def predict_churn(customer_data):
    model_pipeline = joblib.load("bank_customer_churned.pkl")
    featured_columns = joblib.load("bank_customer_churned_featured_columns.pkl")

    customer_df = pd.DataFrame([customer_data])

    for feature in featured_columns:
        if feature not in customer_df.columns:
            raise ValueError(f"Misssing require {feature}")
    customer_df = customer_df[featured_columns]

    prediction = model_pipeline.predict(customer_df)[0]
    probability = model_pipeline.predict_proba(customer_df)[0]

    result = {
        "prediction": "Churned" if prediction == 1 else "Retained",
        "churned_probability": probability[1],
        "retained_probability": probability[0],
        "risk_level": "High" if probability[1] > 0.7 else "Medium" if probability[0] > 0.4 else "Low"
    }

    return result

In [51]:
def test_predict():
    print("\n" + "-" * 60)
    print("TESTING PREDICTION")
    print("-" * 60)

    customer_data = {
        "CreditScore": 400,
        "Geography": "Gammany",
        "Gender": "Female", 
        "Age": 45, 
        "Tenure": 2, 
        "Balance": 100000.0, 
        "NumOfProducts": 1, 
        "HasCrCard": "No", 
        "IsActiveMember": "No",
        "EstimatedSalary": 8000.0,
        "ServiceRating": 1
    }

    customer_data_1 = {
        "CreditScore": 800,
        "Geography": "France",
        "Gender": "Male", 
        "Age": 25, 
        "Tenure": 4, 
        "Balance": 100000.0, 
        "NumOfProducts": 1, 
        "HasCrCard": "Yes", 
        "IsActiveMember": "Yes",
        "EstimatedSalary": 8000.0,
        "ServiceRating": 3
    }

    print("\nCustomer 1")
    result = predict_churn(customer_data)
    print("Prediction -", result["prediction"])
    print("Churned Probability -", result["churned_probability"])
    print("Retained Probability -", result["retained_probability"])
    print("Risk Level -", result["risk_level"])

    print("\nCustomer 2")
    result_1 = predict_churn(customer_data_1)
    print("Prediction -", result_1["prediction"])
    print("Churned Probability -", result_1["churned_probability"])
    print("Retained Probability -", result_1["retained_probability"])
    print("Risk Level -", result_1["risk_level"])

In [54]:
def main():
    file_path = "cleaned_bank_customer_churn.csv"

    df = load_and_explore_dataset(file_path)

    feature_to_drop, num_features, cat_features = identify_features(df)

    x, y, featured_columns = prepare_data(df, num_features, cat_features, feature_to_drop)

    preprocessing = create_preprocessing_pipeline(num_features, cat_features)

    x_train, x_test, y_train, y_test = split_data(x, y)

    model_pipeline = create_pipeline(preprocessing)

    model_pipeline = train(model_pipeline, x_train, y_train)

    metrics = evaluate_model(model_pipeline, x_train, x_test, y_train, y_test)

    save_model_artifact(model_pipeline, featured_columns)

    test_predict()

In [55]:
if __name__ == "__main__":
    main()

------------------------------------------------------------
LOAD AND EXPLORE DATE
------------------------------------------------------------
shape of the dataset
(9997, 14)

check missing
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
EstimatedSalary    0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
Exited             0
ServiceRating      0
dtype: int64

First five row
   CustomerId   Surname  CreditScore Geography  Gender   Age  Tenure  \
0    15634602  Hargrave          619    France  Female  42.0       2   
1    15647311      Hill          608     Spain  Female  41.0       1   
2    15619304      Onio          502    France  Female  42.0       8   
3    15701354      Boni          699    France  Female  39.0       1   
4    15737888  Mitchell          850     Spain  Female  43.0       2   

   EstimatedSalary    Balance  NumOfProducts HasCrCar

C:\Users\MARYAM\Desktop\data-science-class\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
C:\Users\MARYAM\Desktop\data-science-class\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [0] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
